# Synthetic EEG Generator — Tutorial

This notebook walks through the `Pandas_Gen` synthetic EEG package:

| Section | Topic |
|---------|-------|
| 1 | Running `main.py` from the command line |
| 2 | Generating signals with the Python API |
| 3 | Plotting (continuous, PSD, epochs) |
| 4 | Epochs: slicing and labelling trials |
| 5 | Saving generated signals to disk |
| 6 | Controlling the signal via the YAML config file |

> **Prerequisites:** run this notebook with the `Human` conda environment kernel.

In [ ]:
import sys
import os
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams['figure.dpi'] = 100

sys.path.insert(0, os.path.abspath('.'))

from synthetic_eeg import EEGGenerator, plot_continuous, plot_psd, plot_epochs
from synthetic_eeg.io import load_config

---
## 1. Command-Line Interface

`main.py` is the entry point for terminal use. It accepts three optional flags:

| Flag | Description |
|------|-------------|
| `--config PATH` | Path to YAML config (default: `synthetic_eeg/config/default.yaml`) |
| `--save` | Write output to the path in `output.path` |
| `--plot` | Open interactive matplotlib windows (terminal only — see note below) |

In [ ]:
!conda run -n Human python main.py --help

In [ ]:
# Generate continuous + epoch data and save to output/synthetic_eeg.csv
!conda run -n Human python main.py --save

### Plotting from the terminal

The `--plot` flag calls `plt.show()` which opens **interactive windows** — ideal from a terminal, but blocking inside a notebook. Use it from a shell like this:

```bash
# Show plots only
python main.py --plot

# Generate, save, and show plots
python main.py --save --plot

# Use a custom config, save, and show plots
python main.py --config my_config.yaml --save --plot
```

Sections 3–4 below reproduce the same plots inline using the Python API.

---
## 2. Generating Signals with the Python API

`EEGGenerator` takes a config file path and exposes two generation methods:

- `generate()` → continuous `pd.DataFrame` indexed by time (seconds), one column per channel
- `generate_epochs()` → `(X, y)` where `X` is `(n_trials, n_time, n_channels)` and `y` is `(n_trials,)`

In [ ]:
gen = EEGGenerator("synthetic_eeg/config/default.yaml")
df  = gen.generate()

fs = gen.cfg["global"]["fs"]
print(f"Shape   : {df.shape}  ({df.shape[0]} samples × {df.shape[1]} channels)")
print(f"Fs      : {fs} Hz")
print(f"Duration: {gen.cfg['global']['duration']} s")
print(f"Channels: {list(df.columns)}")
df.head()

---
## 3. Plotting

Three plot functions are provided, each returning a `matplotlib.Figure`:

- `plot_continuous(df)` — one subplot per channel, shared time axis
- `plot_psd(df, fs)` — FFT-based power spectral density in dB
- `plot_epochs(X, y, channel_names, fs)` — individual trials overlaid with per-class means

In [ ]:
fig = plot_continuous(df, title="Synthetic EEG — Continuous Signal")
plt.show()

In [ ]:
fig = plot_psd(df, fs=fs, title="Power Spectral Density")
plt.show()

---
## 4. Epochs

When `epochs.enabled: true` in the config, `generate_epochs()` slices the continuous signal into fixed-length windows and assigns class labels (shuffled by default).

In [ ]:
X, y = gen.generate_epochs()

print(f"X shape : {X.shape}  (trials × time-samples × channels)")
print(f"y shape : {y.shape}")
label_counts = {int(v): int((y == v).sum()) for v in sorted(set(y.tolist()))}
print(f"Labels  : {label_counts}")

In [ ]:
fig = plot_epochs(X, y, channel_names=list(df.columns), fs=fs)
plt.show()

---
## 5. Saving Signals

### 5.1 Continuous data

`gen.save(df)` writes the DataFrame to disk using the `format` and `path` fields from the config.
Supported formats: `csv` (default) and `hdf5` (requires the `tables` package).

In [ ]:
gen.save(df)
output_path = gen.cfg["output"]["path"]
print(f"Saved to: {output_path}")

# Reload and verify
df_reloaded = pd.read_csv(output_path, index_col=0)
print(f"Reloaded: {df_reloaded.shape}")
df_reloaded.head()

### 5.2 Saving to a custom path or format

You can also call the underlying I/O helpers directly, or change the config before saving.

In [ ]:
from synthetic_eeg.io import save_csv

# Save to an explicit path (independent of config)
save_csv(df, "./output/my_eeg.csv")
print("Saved to ./output/my_eeg.csv")

# Or override the output path via the config dict before calling gen.save()
gen.cfg["output"]["path"] = "./output/custom_path.csv"
gen.save(df)
print(f"Also saved to: {gen.cfg['output']['path']}")

### 5.3 Epoch arrays

`generate_epochs()` returns plain numpy arrays — save with `np.save` or `np.savez`.

In [ ]:
import pathlib

out = pathlib.Path("./output")
out.mkdir(exist_ok=True)

# Save as separate .npy files
np.save(out / "epochs_X.npy", X)
np.save(out / "epochs_y.npy", y)
print(f"X → {out / 'epochs_X.npy'}  {X.shape}")
print(f"y → {out / 'epochs_y.npy'}  {y.shape}")

# Or bundle both into a single .npz archive
np.savez(out / "epochs.npz", X=X, y=y)
print(f"\nBundle → {out / 'epochs.npz'}")

# Round-trip verification
arcv = np.load(out / "epochs.npz")
assert np.array_equal(arcv["X"], X) and np.array_equal(arcv["y"], y)
print("Round-trip verification passed.")

---
## 6. Controlling the Signal via Config

The YAML config has four top-level sections:

| Section | What it controls |
|---------|------------------|
| `global` | `seed`, `duration` (s), `fs` (Hz) |
| `channels` | List of channel defs — signal components, noise, artifacts |
| `epochs` | Window length, overlap, label counts |
| `output` | File format (`csv` / `hdf5`) and save path |

Each **channel** has three sub-sections:

```yaml
signal:
  components:
    - {freq: 10.0, amplitude: 20.0, phase: 0.0}           # active full duration
    - {freq: 2.0,  amplitude: 50.0, phase: 0.0, start: 0.0, stop: 5.0}  # active only 0–5 s

noise:
  white:  {enabled: true,  amplitude: 2.0}
  pink:   {enabled: true,  amplitude: 5.0}
  drift:  {enabled: true,  amplitude: 10.0, freq: 0.5}

artifacts:
  eye_blink:    {enabled: true,  rate: 0.2,   amplitude: 100.0}
  heartrate:    {enabled: true,  bpm: 74,     amplitude: 5.0}
  muscle_burst: {enabled: false, rate: 0.05,  amplitude: 30.0}
  dead_channel: {enabled: false}
  bad_channel:  {enabled: false, amplitude_multiplier: 5.0}
```

In [ ]:
# Print the default config
with open("synthetic_eeg/config/default.yaml") as f:
    print(f.read())

### 6.1 Quick experiments: modify `gen.cfg` in-place

Load a generator, edit the config dict directly, then call `generate()` again — no file writes needed.
This is the fastest way to iterate during exploration.

In [ ]:
gen2 = EEGGenerator("synthetic_eeg/config/default.yaml")

# ── Global settings ──────────────────────────────────────────────────────────
gen2.cfg["global"]["duration"] = 5.0
gen2.cfg["global"]["seed"]     = 99

# ── Fp1: replace signal with strong alpha + light beta, remove all noise ─────
ch0 = gen2.cfg["channels"][0]
ch0["signal"]["components"] = [
    {"freq": 10.0, "amplitude": 60.0, "phase": 0.0},  # dominant alpha
    {"freq": 20.0, "amplitude": 10.0, "phase": 0.5},  # light beta
]
for key in ("white", "pink", "drift"):
    ch0["noise"][key]["enabled"] = False
for key in ("eye_blink", "heartrate", "muscle_burst"):
    ch0["artifacts"][key]["enabled"] = False

# ── Fp3: enable a muscle burst artifact ──────────────────────────────────────
ch2 = gen2.cfg["channels"][2]
ch2["artifacts"]["muscle_burst"]["enabled"]   = True
ch2["artifacts"]["muscle_burst"]["rate"]      = 0.1
ch2["artifacts"]["muscle_burst"]["amplitude"] = 40.0

df2 = gen2.generate()

fig = plot_continuous(df2, title="Modified Config — Fp1: Clean Alpha+Beta; Fp3: With Muscle Bursts")
plt.show()

fig2 = plot_psd(df2, fs=gen2.cfg["global"]["fs"],
                title="PSD — Clean 10 Hz and 20 Hz peaks visible on Fp1")
plt.show()

### 6.2 Writing a reusable config file

Build the config as a Python dict, dump it to YAML with `yaml.dump`, then pass the path to `EEGGenerator`.
This is the preferred approach for reproducible, version-controlled experiments.

In [ ]:
# A motor-imagery style config: two sensorimotor channels (C3 / C4)
mi_cfg = {
    "global": {
        "seed": 7,
        "duration": 10.0,
        "fs": 512,            # higher sampling rate
    },
    "channels": [
        {
            "name": "C3",
            "signal": {
                "components": [
                    {"freq": 8.0,  "amplitude": 30.0, "phase": 0.0},   # low alpha / mu
                    {"freq": 12.0, "amplitude": 20.0, "phase": 0.0},   # SMR
                ]
            },
            "noise": {
                "white": {"enabled": True,  "amplitude": 3.0},
                "pink":  {"enabled": True,  "amplitude": 8.0},
                "drift": {"enabled": False, "amplitude": 10.0, "freq": 0.1},
            },
            "artifacts": {
                "eye_blink":    {"enabled": False, "rate": 0.2,  "amplitude": 100.0},
                "heartrate":    {"enabled": True,  "bpm": 68,    "amplitude": 4.0},
                "muscle_burst": {"enabled": False, "rate": 0.05, "amplitude": 30.0},
                "dead_channel": {"enabled": False},
                "bad_channel":  {"enabled": False, "amplitude_multiplier": 5.0},
            },
        },
        {
            "name": "C4",
            "signal": {
                "components": [
                    {"freq": 10.0, "amplitude": 25.0, "phase": 0.1},   # alpha
                    {"freq": 25.0, "amplitude": 10.0, "phase": 0.0},   # low beta
                ]
            },
            "noise": {
                "white": {"enabled": True, "amplitude": 2.0},
                "pink":  {"enabled": True, "amplitude": 6.0},
                "drift": {"enabled": True, "amplitude": 8.0, "freq": 0.2},
            },
            "artifacts": {
                "eye_blink":    {"enabled": False, "rate": 0.2,  "amplitude": 100.0},
                "heartrate":    {"enabled": True,  "bpm": 68,    "amplitude": 4.0},
                "muscle_burst": {"enabled": True,  "rate": 0.04, "amplitude": 22.0},
                "dead_channel": {"enabled": False},
                "bad_channel":  {"enabled": False, "amplitude_multiplier": 5.0},
            },
        },
    ],
    "epochs": {
        "enabled": True,
        "length": 2.0,         # 2-second trials
        "overlap": 0.5,        # 0.5 s overlap between consecutive trials
        "shuffle": True,
        "labels": [
            {"class": 0, "count": 15},  # e.g. left-hand imagery
            {"class": 1, "count": 15},  # e.g. right-hand imagery
        ],
    },
    "output": {
        "format": "csv",
        "path": "./output/motor_imagery.csv",
    },
}

mi_path = "./motor_imagery_config.yaml"
with open(mi_path, "w") as f:
    yaml.dump(mi_cfg, f, default_flow_style=False, sort_keys=False)

print(f"Config written to {mi_path}")
print()
with open(mi_path) as f:
    print(f.read())

In [ ]:
# Load the custom config and generate
gen_mi = EEGGenerator(mi_path)
df_mi  = gen_mi.generate()
fs_mi  = gen_mi.cfg["global"]["fs"]

print(f"Shape   : {df_mi.shape}")
print(f"Channels: {list(df_mi.columns)}")
print(f"Fs      : {fs_mi} Hz")

fig = plot_continuous(df_mi, title="Motor-Imagery Config — C3 / C4")
plt.show()

fig2 = plot_psd(df_mi, fs=fs_mi, title="PSD — C3 / C4")
plt.show()

In [ ]:
# Generate epochs, save everything, then plot the epoch overlay
X_mi, y_mi = gen_mi.generate_epochs()
print(f"Epochs  : {X_mi.shape}")

# Save continuous data (path from config)
gen_mi.save(df_mi)
print(f"Continuous saved → {gen_mi.cfg['output']['path']}")

# Save epoch arrays
np.savez("./output/motor_imagery_epochs.npz", X=X_mi, y=y_mi)
print("Epochs saved → ./output/motor_imagery_epochs.npz")

fig3 = plot_epochs(X_mi, y_mi, channel_names=list(df_mi.columns), fs=fs_mi)
plt.show()

### Config parameter reference

#### `global`
| Key | Type | Description |
|-----|------|-------------|
| `seed` | int | NumPy random seed for reproducibility |
| `duration` | float | Length of the continuous signal in seconds |
| `fs` | float | Sampling frequency in Hz |

#### `channels[*].signal.components[*]`
| Key | Type | Description |
|-----|------|-------------|
| `freq` | float | Frequency of the sinusoid (Hz) |
| `amplitude` | float | Peak amplitude (µV) |
| `phase` | float | Phase offset (radians) |
| `start` | float | (optional) Activate component from this time (s) |
| `stop` | float | (optional) Deactivate component at this time (s) |

#### `channels[*].noise`
| Key | Parameters | Description |
|-----|-----------|-------------|
| `white` | `enabled`, `amplitude` | Gaussian white noise |
| `pink` | `enabled`, `amplitude` | 1/f pink noise |
| `drift` | `enabled`, `amplitude`, `freq` | Slow sinusoidal drift |

#### `channels[*].artifacts`
| Key | Parameters | Description |
|-----|-----------|-------------|
| `eye_blink` | `enabled`, `rate` (blinks/s), `amplitude` | Gaussian-pulse blink transients |
| `heartrate` | `enabled`, `bpm`, `amplitude` | Cardiac interference (sinusoid at bpm/60 Hz) |
| `muscle_burst` | `enabled`, `rate`, `amplitude` | High-frequency (30–100 Hz) bursts |
| `dead_channel` | `enabled` | Zeroes the entire channel |
| `bad_channel` | `enabled`, `amplitude_multiplier` | Replaces channel with scaled white noise |

#### `epochs`
| Key | Type | Description |
|-----|------|-------------|
| `enabled` | bool | Must be `true` to call `generate_epochs()` |
| `length` | float | Window length in seconds |
| `overlap` | float | Overlap between consecutive windows (s) |
| `shuffle` | bool | Shuffle label assignment across trials |
| `labels` | list | `[{class: int, count: int}, ...]` |

#### `output`
| Key | Values | Description |
|-----|--------|-------------|
| `format` | `csv`, `hdf5` | File format (`hdf5` requires `tables`) |
| `path` | string | Output file path |